# Analyzing Electric Vehicle Charging Patterns with LightGBM
Understanding electric vehicle (EV) charging patterns is crucial for optimizing charging infrastructure, enhancing user experience, and managing energy consumption effectively. Machine learning models can provide valuable insights into these patterns, enabling predictive analytics and informed decision-making. This article walks you through a Python script designed to analyze EV charging data using LightGBM, a powerful gradient boosting framework. We will explore each step of the code, explaining the processes and methodologies employed to build a robust predictive model.

---

## Introduction

Electric vehicles are becoming increasingly popular, leading to a growing need for efficient and effective charging infrastructure. Analyzing charging patterns helps in understanding user behavior, optimizing the placement of charging stations, and managing energy distribution. Machine learning models, such as LightGBM, can predict energy consumption and identify trends, facilitating better planning and resource allocation.

In this guide, we will dissect a Python script that processes EV charging data, performs feature engineering, handles missing values, and trains a LightGBM regression model to predict energy cosumption during charging sessions.


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/electric-vehicle-charging-patterns/ev_charging_patterns.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

# ---------------------------------------------------
# Step 1: Load the Data
# ---------------------------------------------------

# Load the dataset from a CSV file into a pandas DataFrame
# The dataset is assumed to be located at '/kaggle/input/electric-vehicle-charging-patterns/ev_charging_patterns.csv'
df = pd.read_csv("/kaggle/input/electric-vehicle-charging-patterns/ev_charging_patterns.csv")

# ---------------------------------------------------
# Step 2: Convert Datetime Columns and Extract Features
# ---------------------------------------------------

# Convert 'Charging Start Time' column to datetime objects for easier manipulation
df["Charging Start Time"] = pd.to_datetime(df["Charging Start Time"])

# Convert 'Charging End Time' column to datetime objects
df["Charging End Time"] = pd.to_datetime(df["Charging End Time"])

# Extract additional temporal features from 'Charging Start Time'

# Extract the hour (0-23) when the charging session started
df["Hour"] = df["Charging Start Time"].dt.hour

# Extract the month (1-12) when the charging session started
df["Month"] = df["Charging Start Time"].dt.month

# Extract the year when the charging session started
df["Year"] = df["Charging Start Time"].dt.year

# ---------------------------------------------------
# Step 3: Add Charging Duration Feature
# ---------------------------------------------------

# Calculate the duration of each charging session in hours
# This is done by subtracting 'Charging Start Time' from 'Charging End Time'
# The result is converted from seconds to hours
df["Charging Duration"] = (
    df["Charging End Time"] - df["Charging Start Time"]
).dt.total_seconds() / 3600

# ---------------------------------------------------
# Step 4: Add Charging Rate Feature
# ---------------------------------------------------

# Calculate the average charging rate (kW) for each session
# Charging Rate = Energy Consumed (kWh) / Charging Duration (hours)
df["Charging Rate (kW)"] = df["Energy Consumed (kWh)"] / df["Charging Duration"]

# ---------------------------------------------------
# Step 5: Create Interaction Features
# ---------------------------------------------------

# Create new features by interacting existing features to capture combined effects

# Interaction between Charging Duration and Hour
df["Duration_Hour"] = df["Charging Duration"] * df["Hour"]

# Interaction between Charging Duration and Charging Rate
df["Duration_Rate"] = df["Charging Duration"] * df["Charging Rate (kW)"]

# Interaction between Hour and Charging Rate
df["Hour_Rate"] = df["Hour"] * df["Charging Rate (kW)"]

# ---------------------------------------------------
# Step 6: Handle Missing Values
# ---------------------------------------------------

# Identify numerical columns by selecting columns with float data types
numerical_columns = df.select_dtypes(include=["float64"]).columns

# Initialize a SimpleImputer to fill missing values with the median of each column
imputer = SimpleImputer(strategy="median")

# Apply the imputer to the numerical columns
df[numerical_columns] = imputer.fit_transform(df[numerical_columns])

# ---------------------------------------------------
# Step 7: Replace Infinite Values with Median
# ---------------------------------------------------

# Iterate over each numerical column to replace infinite values
for col in numerical_columns:
    # Calculate the median value of the column
    median_val = df[col].median()
    
    # Replace positive and negative infinite values with the median
    df[col] = df[col].replace([np.inf, -np.inf], median_val)

# ---------------------------------------------------
# Step 8: Encode Categorical Columns
# ---------------------------------------------------

# List of categorical columns that need to be label encoded
categorical_columns = [
    "Charger Type",
    "Charging Station ID",
    "Charging Station Location",
    "Day of Week",
    "Time of Day",
    "User ID",
    "User Type",
    "Vehicle Model",
]

# Initialize a dictionary to store LabelEncoders for each categorical column
label_encoders = {}

# Iterate over each categorical column to apply label encoding
for col in categorical_columns:
    # Initialize a LabelEncoder for the current column
    label_encoders[col] = LabelEncoder()
    
    # Fit the LabelEncoder on the column data and transform it
    df[col] = label_encoders[col].fit_transform(df[col])

# ---------------------------------------------------
# Step 9: Prepare Features and Target Variable
# ---------------------------------------------------

# Define the list of features by excluding target and irrelevant columns
# Exclude 'Energy Consumed (kWh)', 'Charging End Time', and 'Charging Start Time' as they are not used for prediction
features = [
    col
    for col in df.columns
    if col not in ["Energy Consumed (kWh)", "Charging End Time", "Charging Start Time"]
]

# Assign features to X and target variable to y
X = df[features]
y = df["Energy Consumed (kWh)"]

# ---------------------------------------------------
# Step 10: Split the Data into Training and Validation Sets
# ---------------------------------------------------

# Split the dataset into training and validation sets
# 80% of the data is used for training and 20% for validation
# random_state=42 ensures reproducibility of the split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------------
# Step 11: Create LightGBM Datasets
# ---------------------------------------------------

# Create a LightGBM Dataset for training
# Specify categorical features to let LightGBM handle them appropriately
train_data = lgb.Dataset(
    X_train, label=y_train, categorical_feature=categorical_columns
)

# Create a LightGBM Dataset for validation
val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_columns)

# ---------------------------------------------------
# Step 12: Set LightGBM Parameters
# ---------------------------------------------------

# Define a dictionary of parameters for the LightGBM model
params = {
    "objective": "regression",           # Specifies the learning task and the corresponding learning objective
    "metric": "rmse",                    # Root Mean Squared Error as the evaluation metric
    "boosting_type": "gbdt",             # Gradient Boosting Decision Tree
    "num_leaves": 31,                    # Maximum number of leaves in one tree
    "learning_rate": 0.05,               # Step size shrinkage used in update to prevent overfitting
    "feature_fraction": 0.9,             # Fraction of features to consider when building each tree
    "verbose": -1                        # Suppress verbose output
}

# ---------------------------------------------------
# Step 13: Train the LightGBM Model
# ---------------------------------------------------

# Train the LightGBM model using the training data
model = lgb.train(
    params,                             # Model parameters
    train_data,                         # Training data
    valid_sets=[train_data, val_data],  # Validation sets for monitoring
    num_boost_round=100                 # Number of boosting iterations
)

# ---------------------------------------------------
# Step 14: Make Predictions and Calculate RMSE on Validation Set
# ---------------------------------------------------

# Make predictions on the validation set using the trained model
val_predictions = model.predict(X_val)

# Calculate the Root Mean Squared Error (RMSE) between actual and predicted values
rmse = np.sqrt(mean_squared_error(y_val, val_predictions))

# ---------------------------------------------------
# Step 15: Perform Cross-Validation
# ---------------------------------------------------

# Perform 5-fold cross-validation to evaluate the model's performance
cv_scores = lgb.cv(
    params,                                              # Model parameters
    lgb.Dataset(X, label=y, categorical_feature=categorical_columns),  # Entire dataset as LightGBM Dataset
    num_boost_round=100,                                 # Number of boosting iterations
    nfold=5,                                             # Number of cross-validation folds
    stratified=False,                                    # Do not perform stratified sampling
    metrics=["rmse"]                                     # Evaluation metric
)

# Get the final RMSE score (changed the key to 'valid rmse-mean')
# LightGBM's cv function returns the mean and standard deviation of the metric for each round
# 'valid rmse-mean' corresponds to the average RMSE across the validation folds
cv_rmse = np.sqrt(cv_scores['valid rmse-mean'][-1])

# ---------------------------------------------------
# Step 16: Print Validation and Cross-Validation RMSE
# ---------------------------------------------------

# Print the RMSE on the validation set
print(f"Validation RMSE: {rmse:.4f}")

# Print the final RMSE from cross-validation
print(f"5-fold Cross-Validation RMSE: {cv_rmse:.4f}")

Validation RMSE: 2.8419
5-fold Cross-Validation RMSE: 1.3914



## Conclusion

This Python script provides a comprehensive approach to analyzing electric vehicle charging patterns using LightGBM, a powerful gradient boosting framework. By following a structured workflow encompassing data preprocessing, feature engineering, handling missing values, and model training with cross-validation, the script ensures the development of an accurate and reliable predictive model.

**Key Takeaways**:

1. **Data Preprocessing**:
   - **Datetime Conversion and Feature Extraction**: Transforming datetime columns into meaningful temporal features enhances the model's ability to capture time-dependent patterns.
   - **Calculating Additional Metrics**: Features like charging duration and charging rate provide deeper insights into charging behaviors.
   - **Creating Interaction Features**: Combining multiple features can reveal complex relationships that improve model performance.
   - **Handling Missing and Infinite Values**: Ensuring data quality by imputing missing values and replacing infinite values is crucial for accurate modeling.

2. **Feature Selection**:
   - **Categorical vs. Numerical Features**: Differentiating between categorical and numerical features allows for appropriate preprocessing steps like encoding and scaling.

3. **Model Training and Evaluation**:
   - **Splitting Data**: Dividing the dataset into training and validation sets enables the assessment of model performance on unseen data.
   - **LightGBM Datasets**: Utilizing LightGBM's optimized data structures facilitates efficient training.
   - **Setting Parameters**: Carefully selecting model parameters influences the model's ability to learn and generalize.
   - **Cross-Validation**: Implementing cross-validation provides a more reliable estimate of model performance, mitigating overfitting risks.

4. **Performance Metrics**:
   - **RMSE**: A standard regression metric that quantifies the model's prediction accuracy. Lower RMSE values indicate better performance.

**Potential Improvements**:

- **Hyperparameter Tuning**: Exploring different hyperparameters using techniques like grid search or Bayesian optimization could further enhance model performance.
- **Advanced Feature Engineering**: Incorporating more complex features, such as lagged variables or aggregated statistics, might capture additional patterns.
- **Handling Categorical Variables**: Utilizing more sophisticated encoding techniques, such as target encoding or one-hot encoding, could improve the model's ability to interpret categorical data.
- **Model Ensemble**: Combining LightGBM with other models in an ensemble could leverage their collective strngths for better predictions.
